# 04 · 跨表特征合并与数据质量校验

**目标：** 将上游聚合到申请级的特征表按 `case_id` 合并，形成 Train/Test 建模宽表，并验证合并后的结构、主键和数据质量。

**处理流程**

```text
核心申请与征信表聚合结果 ─┐
账户与人员表聚合结果     ├─→ base LEFT JOIN → Train/Test 特征宽表
人员明细与税务表聚合结果 ─┘
```

本阶段只执行特征合并、schema 一致性校验、字段血缘汇总及质量审计，不执行缺失值填补、特征选择或模型训练。下方保留原实验的运行输出。


## 输入依赖

Kaggle Inputs 需包含以下三份上游输出：

1. `home-credit-3-single-table-aggregation-logic`
2. `风险模型-3表内聚合-第二部分-A-ZJing`
3. `风险模型-3表内聚合-第二部分-B-ZJing`

输入格式为上游生成的 parquet 文件。

In [25]:
from pathlib import Path
import gc
import polars as pl

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/joined")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

OUT_TRAIN = OUTPUT_ROOT / "home_credit_features_train.parquet"
OUT_TEST = OUTPUT_ROOT / "home_credit_features_test.parquet"

print("Polars：", pl.__version__)
print("输出目录：", OUTPUT_ROOT)

Polars： 1.35.2
输出目录： /kaggle/working/joined


## 1. 输入定位

按文件名递归检索 `/kaggle/input/`，不依赖 Kaggle 自动生成的目录名。

In [26]:
def locate(filename: str) -> Path:
    """在所有 Kaggle Inputs 中精确定位一个文件。"""
    matches = list(INPUT_ROOT.rglob(filename))
    assert len(matches) == 1, (
        f"{filename} 应只找到 1 个，实际找到 {len(matches)} 个：{matches}"
    )
    return matches[0]

print("当前添加的 Input 目录：")
for path in sorted(p for p in INPUT_ROOT.iterdir() if p.is_dir()):
    print(" -", path)

当前添加的 Input 目录：
 - /kaggle/input/competitions
 - /kaggle/input/notebooks


## 2. 特征表清单

`base` 为主表；16 张右表均应聚合至每个 `case_id` 最多一行。

`person_2_person_level.parquet` 为中间层明细，不参与合并；使用最终的 `person_2_agg.parquet`。

In [27]:
BASE_FILES = {
    "train": "base_clean_train.parquet",
    "test": "base_clean_test.parquet",
}

FEATURE_TABLES = [
    # 核心申请与征信模块：8 张表
    {"name": "static_0",          "train": "static_0_clean_train.parquet",          "test": "static_0_clean_test.parquet"},
    {"name": "static_cb_0",       "train": "static_cb_0_clean_train.parquet",       "test": "static_cb_0_clean_test.parquet"},
    {"name": "applprev_1",        "train": "applprev_1_clean_train.parquet",        "test": "applprev_1_clean_test.parquet"},
    {"name": "applprev_2",        "train": "applprev_2_clean_train.parquet",        "test": "applprev_2_clean_test.parquet"},
    {"name": "credit_bureau_a_1", "train": "credit_bureau_a_1_clean_train.parquet", "test": "credit_bureau_a_1_clean_test.parquet"},
    {"name": "credit_bureau_a_2", "train": "credit_bureau_a_2_clean_train.parquet", "test": "credit_bureau_a_2_clean_test.parquet"},
    {"name": "credit_bureau_b_1", "train": "credit_bureau_b_1_clean_train.parquet", "test": "credit_bureau_b_1_clean_test.parquet"},
    {"name": "credit_bureau_b_2", "train": "credit_bureau_b_2_clean_train.parquet", "test": "credit_bureau_b_2_clean_test.parquet"},

    # 账户、人员与税务模块：8 张表
    {"name": "debitcard_1",        "train": "train_debitcard_1_agg.parquet",        "test": "test_debitcard_1_agg.parquet"},
    {"name": "deposit_1",          "train": "train_deposit_1_agg.parquet",          "test": "test_deposit_1_agg.parquet"},
    {"name": "other_1",            "train": "train_other_1_agg.parquet",            "test": "test_other_1_agg.parquet"},
    {"name": "person_1",           "train": "train_person_1_agg.parquet",           "test": "test_person_1_agg.parquet"},
    {"name": "person_2",           "train": "train_person_2_agg.parquet",           "test": "test_person_2_agg.parquet"},
    {"name": "tax_registry_a_1",   "train": "train_tax_registry_a_1_agg.parquet",   "test": "test_tax_registry_a_1_agg.parquet"},
    {"name": "tax_registry_b_1",   "train": "train_tax_registry_b_1_agg.parquet",   "test": "test_tax_registry_b_1_agg.parquet"},
    {"name": "tax_registry_c_1",   "train": "train_tax_registry_c_1_agg.parquet",   "test": "test_tax_registry_c_1_agg.parquet"},
]

BASE_PATHS = {split: locate(filename) for split, filename in BASE_FILES.items()}
for table in FEATURE_TABLES:
    table["train_path"] = locate(table["train"])
    table["test_path"] = locate(table["test"])

print(f"输入文件检查通过：Base 2 个，特征表 {len(FEATURE_TABLES)} 对")

输入文件检查通过：Base 2 个，特征表 16 对


## 3. 合并约束检查

校验 Base 与各特征表的 Train/Test 字段结构，并记录 Test 中需要转换为 Train dtype 的兼容字段。字段缺失、顺序变化或不兼容类型将终止运行。

In [28]:
def parquet_schema(path: Path):
    """
    读取 Parquet 文件的字段结构。

    只获取字段名称、顺序和数据类型，
    不加载完整数据表，适合宽表的结构检查。
    """

    return (
        # 创建惰性扫描，不立即读取全部数据
        pl.scan_parquet(path)

        # 从 Parquet 元数据中获取完整 Schema
        .collect_schema()
    )


# Base 只允许 Train 多出 target
base_train_schema = parquet_schema(
    BASE_PATHS["train"]
)
base_test_schema = parquet_schema(
    BASE_PATHS["test"]
)

assert [
    (column, dtype)
    for column, dtype
    in base_train_schema.items()
    if column != "target"
] == list(base_test_schema.items()), (
    "Base Train/Test schema 不一致"
)

assert (
    "target" in base_train_schema
    and "target" not in base_test_schema
)


numeric_dtypes = {
    pl.Int8,
    pl.Int16,
    pl.Int32,
    pl.Int64,
    pl.UInt8,
    pl.UInt16,
    pl.UInt32,
    pl.UInt64,
    pl.Float32,
    pl.Float64,
}


for table in FEATURE_TABLES:
    train_schema = parquet_schema(
        table["train_path"]
    )
    test_schema = parquet_schema(
        table["test_path"]
    )

    assert (
        list(train_schema.keys())
        == list(test_schema.keys())
    ), (
        f"{table['name']} Train/Test "
        "字段名或顺序不一致"
    )

    dtype_differences = {
        column: (
            train_schema[column],
            test_schema[column],
        )
        for column in train_schema
        if (
            train_schema[column]
            != test_schema[column]
        )
    }

    invalid_differences = [
        (column, train_dtype, test_dtype)
        for column, (
            train_dtype,
            test_dtype,
        ) in dtype_differences.items()
        if not (
            test_dtype == pl.Null
            or (
                train_dtype in numeric_dtypes
                and test_dtype in numeric_dtypes
            )
        )
    ]

    assert not invalid_differences, (
        f"{table['name']} 存在不兼容类型："
        f"{invalid_differences[:10]}"
    )

    table["test_casts"] = {
        column: train_dtype
        for column, (
            train_dtype,
            test_dtype,
        ) in dtype_differences.items()
    }


print("合并约束检查通过")

合并约束检查通过


## 4. Train/Test 宽表构建

以 Base 为左表，Train/Test 使用相同的右表顺序。采用 Polars LazyFrame 生成执行计划，未匹配的右表特征保留为空。

In [29]:
def build_join_plan(split: str) -> pl.LazyFrame:
    assert split in {"train", "test"}

    result = (
        pl.scan_parquet(BASE_PATHS[split])
        .with_columns(
            pl.col("case_id").cast(pl.UInt32)
        )
    )

    for table in FEATURE_TABLES:
        right = (
            pl.scan_parquet(
                table[f"{split}_path"]
            )
            .with_columns(
                pl.col("case_id").cast(pl.UInt32)
            )
        )

        # 将 Test 的兼容 dtype 转换成对应的 Train dtype
        if split == "test":
            test_casts = table.get(
                "test_casts",
                {},
            )

            if test_casts:
                right = right.with_columns([
                    pl.col(column)
                    .cast(dtype, strict=False)
                    .alias(column)
                    for column, dtype
                    in test_casts.items()
                ])

        # 在 for table 循环内，但不属于 if split
        result = result.join(
            right,
            on="case_id",
            how="left",
        )

    return result

## 5. 宽表输出

结果写入 `/kaggle/working/joined/`，供下游建模流程读取。

In [30]:
print("构建 Train Join 计划……")
train_plan = build_join_plan("train")

print("开始生成 Train 总特征表……")
train_plan.sink_parquet(
    OUT_TRAIN,
    compression="zstd",
    compression_level=3,
)
print("Train 已保存：", OUT_TRAIN)

del train_plan
gc.collect()


print("构建 Test Join 计划……")
test_plan = build_join_plan("test")

print("开始生成 Test 总特征表……")
test_plan.sink_parquet(
    OUT_TEST,
    compression="zstd",
    compression_level=3,
)
print("Test 已保存：", OUT_TEST)

del test_plan
gc.collect()

构建 Train Join 计划……
开始生成 Train 总特征表……
Train 已保存： /kaggle/working/joined/home_credit_features_train.parquet
构建 Test Join 计划……
开始生成 Test 总特征表……
Test 已保存： /kaggle/working/joined/home_credit_features_test.parquet


0

## 6. 结果校验

校验 Base 行数守恒、`case_id` 唯一、Train/Test schema 对齐，以及 `target` 仅存在于 Train。

In [31]:
# Join 前的 Base 主键，用于与最终宽表核对行数
base_keys = {
    "train": pl.read_parquet(
        BASE_PATHS["train"],
        columns=["case_id"],
    ),
    "test": pl.read_parquet(
        BASE_PATHS["test"],
        columns=["case_id"],
    ),
}
train_keys_final = pl.read_parquet(OUT_TRAIN, columns=["case_id"])
test_keys_final = pl.read_parquet(OUT_TEST, columns=["case_id"])

assert train_keys_final.height == base_keys["train"].height, "Train Join 后行数发生变化"
assert test_keys_final.height == base_keys["test"].height, "Test Join 后行数发生变化"
assert train_keys_final["case_id"].n_unique() == train_keys_final.height, "Train case_id 不唯一"
assert test_keys_final["case_id"].n_unique() == test_keys_final.height, "Test case_id 不唯一"

final_train_schema = parquet_schema(OUT_TRAIN)
final_test_schema = parquet_schema(OUT_TEST)
final_train_without_target = [(c, dt) for c, dt in final_train_schema.items() if c != "target"]

assert final_train_without_target == list(final_test_schema.items()), (
    "最终 Train/Test 的列名、顺序或 dtype 不一致"
)
assert "target" in final_train_schema and "target" not in final_test_schema

train_size_mb = OUT_TRAIN.stat().st_size / 1024**2
test_size_mb = OUT_TEST.stat().st_size / 1024**2

print("=" * 60)
print("最终 Join 检查通过")
print(f"Train：{train_keys_final.height:,} 行 × {len(final_train_schema):,} 列，{train_size_mb:.2f} MB")
print(f"Test ：{test_keys_final.height:,} 行 × {len(final_test_schema):,} 列，{test_size_mb:.2f} MB")
print("Train 输出：", OUT_TRAIN)
print("Test  输出：", OUT_TEST)
print("=" * 60)

最终 Join 检查通过
Train：1,526,659 行 × 1,457 列，1022.00 MB
Test ：10 行 × 1,456 列，0.54 MB
Train 输出： /kaggle/working/joined/home_credit_features_train.parquet
Test  输出： /kaggle/working/joined/home_credit_features_test.parquet


## 7. 最终特征结构与宽表预览

从字段明细、来源构成和样本记录三个视角检查最终宽表：

- 纵向展示全部字段的名称、来源、类型和角色；
- 汇总各上游表贡献的模型特征数量；
- 横向展示 Train/Test 前 5 行及全部字段。

### 7.1 纵向特征目录

将最终宽表转换为“一行一个字段”的目录，记录字段位置、名称、来源、数据类型和字段角色。

该目录用于查看完整字段清单、追踪特征来源，并为后续日期转换、类别编码和数值处理提供结构信息。

In [32]:
from IPython.display import HTML, display


# Base 字段来源
source_map = {
    column: "base"
    for column in base_train_schema.keys()
}


# 上游特征表字段来源
for table in FEATURE_TABLES:
    table_schema = parquet_schema(
        table["train_path"]
    )

    for column in table_schema.keys():
        if column != "case_id":
            source_map[column] = table["name"]


# 最终字段目录
feature_catalog = pl.DataFrame({
    "position": list(
        range(1, len(final_train_schema) + 1)
    ),
    "column": list(final_train_schema.keys()),
    "source": [
        source_map.get(column, "unknown")
        for column in final_train_schema.keys()
    ],
    "dtype": [
        str(dtype)
        for dtype in final_train_schema.values()
    ],
    "role": [
        "identifier"
        if column == "case_id"
        else "target"
        if column == "target"
        else "feature"
        for column in final_train_schema.keys()
    ],
})


# 在 Notebook 中以纵向滚动表格展示
catalog_html = (
    feature_catalog
    .to_pandas()
    .to_html(
        index=False,
        border=0,
    )
)

display(HTML(
    "<div style='"
    "max-height:600px;"
    "overflow:auto;"
    "border:1px solid #ddd;"
    "'>"
    + catalog_html
    + "</div>"
))

position,column,source,dtype,role
1,case_id,base,UInt32,identifier
2,date_decision,base,Date,feature
3,MONTH,base,UInt32,feature
4,WEEK_NUM,base,UInt8,feature
5,target,base,UInt8,target
6,actualdpdtolerance_344P,static_0,Float64,feature
7,amtinstpaidbefduel24m_4187115A,static_0,Float64,feature
8,annuity_780A,static_0,Float64,feature
9,annuitynextmonth_57A,static_0,Float64,feature
10,applicationcnt_361L,static_0,Float64,feature


### 7.2 来源特征数量

统计各上游表为最终宽表贡献的模型特征数量，用于了解特征构成和检查数据血缘。

特征数量只表示字段规模，不代表预测能力。实际价值仍需结合交叉验证、特征重要性和稳定性分析判断。

In [33]:
source_summary = (
    feature_catalog
    .filter(
        pl.col("role") == "feature"
    )
    .group_by("source")
    .len()
    .rename({
        "len": "feature_count"
    })
    .sort(
        "feature_count",
        descending=True,
    )
)

display(
    source_summary.to_pandas()
)

,source,feature_count
0,person_2,272
1,credit_bureau_a_1,243
2,applprev_1,203
3,static_0,163
4,credit_bureau_b_1,137
5,person_1,90
6,applprev_2,52
7,static_cb_0,49
8,credit_bureau_a_2,49
9,debitcard_1,41


### 7.3 横向宽表预览

展示 Train/Test 前 5 行及全部字段，用于检查字段排列、合并结果、缺失值和样本值是否存在明显异常。

少量样本不具有统计代表性，不用于判断特征有效性、总体缺失率或数据漂移。

In [34]:
HORIZONTAL_PREVIEW_ROWS = 5

preview_tables = {
    "Train": OUT_TRAIN,
    "Test": OUT_TEST,
}


for name, path in preview_tables.items():
    preview = (
        pl.read_parquet(
            path,
            n_rows=HORIZONTAL_PREVIEW_ROWS,
        )
        .to_pandas()
    )

    preview_html = preview.to_html(
        index=False,
        border=0,
    )

    display(HTML(
        f"<h4>{name}：前 {len(preview)} 行，"
        f"共 {preview.shape[1]} 列</h4>"
        "<div style='"
        "max-width:100%;"
        "max-height:500px;"
        "overflow:auto;"
        "border:1px solid #ddd;"
        "white-space:nowrap;"
        "'>"
        + preview_html
        + "</div>"
    ))

## 8. Schema 汇总

汇总宽表字段类型，并输出完整 schema 明细。

In [35]:
from pathlib import Path

# 审计报告统一保存位置
REPORT_ROOT = Path("/kaggle/working/reports")
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
schema_detail = pl.DataFrame({
    "column": list(final_train_schema.keys()),
    "dtype": [str(dt) for dt in final_train_schema.values()],
})
dtype_summary = (
    schema_detail.group_by("dtype")
    .len()
    .rename({"len": "column_count"})
    .sort("column_count", descending=True)
)

schema_detail.write_csv(REPORT_ROOT / "feature_schema_report.csv")
display(dtype_summary)
print("Schema 明细已保存：", REPORT_ROOT / "feature_schema_report.csv")

dtype,column_count
str,u32
"""Float64""",960
"""UInt16""",119
"""Int64""",75
"""Date""",69
"""UInt32""",68
…,…
"""UInt8""",31
"""Boolean""",15
"""Null""",10


Schema 明细已保存： /kaggle/working/reports/feature_schema_report.csv


## 9. 数据质量报告

统计 Train/Test 缺失率、全空列和无穷值。本次保存输出使用的 Test 为 10 条记录；该规模下的缺失率仅用于排查数据异常，不作为特征删除或漂移判断依据。

In [36]:
FLOAT_DTYPES = {pl.Float32, pl.Float64}

def build_quality_report(path: Path, split: str) -> pl.DataFrame:
    schema = parquet_schema(path)
    n_rows = pl.scan_parquet(path).select(pl.len().alias("n")).collect().item()

    expressions = []
    for i, (column, dtype) in enumerate(schema.items()):
        if dtype in FLOAT_DTYPES:
            missing_expr = (
                pl.col(column).is_null().sum()
                + pl.col(column).is_nan().fill_null(False).sum()
            )
            infinite_expr = pl.col(column).is_infinite().fill_null(False).sum()
        else:
            missing_expr = pl.col(column).is_null().sum()
            infinite_expr = pl.lit(0)

        expressions.append(missing_expr.alias(f"m_{i}"))
        expressions.append(infinite_expr.alias(f"i_{i}"))

    values = (
        pl.scan_parquet(path)
        .select(expressions)
        .collect(engine="streaming")
        .row(0, named=True)
    )

    rows = []
    for i, (column, dtype) in enumerate(schema.items()):
        missing_count = int(values[f"m_{i}"] or 0)
        rows.append({
            "column": column,
            f"{split}_dtype": str(dtype),
            f"{split}_missing_count": missing_count,
            f"{split}_missing_rate": missing_count / n_rows if n_rows else None,
            f"{split}_infinite_count": int(values[f"i_{i}"] or 0),
        })

    return pl.DataFrame(rows)

print("正在扫描最终宽表并统计数据质量，Train 表较大，可能需要几分钟……")
train_quality = build_quality_report(OUT_TRAIN, "train")
test_quality = build_quality_report(OUT_TEST, "test")

quality_report = (
    train_quality.join(test_quality, on="column", how="left")
    .with_columns(
        (pl.col("train_missing_rate") - pl.col("test_missing_rate"))
        .abs()
        .alias("missing_rate_abs_gap")
    )
    .with_columns([
        (pl.col("train_missing_rate") == 1).alias("train_all_missing"),
        (pl.col("test_missing_rate") == 1).alias("test_all_missing"),
        (pl.col("train_missing_rate") >= 0.90).alias("train_missing_ge_90pct"),
    ])
)

quality_report.write_csv(REPORT_ROOT / "feature_quality_report.csv")

print("Train 全空列：", quality_report.filter(pl.col("train_all_missing")).height)
print("Train 缺失率 ≥ 90% 的列：", quality_report.filter(pl.col("train_missing_ge_90pct")).height)
print("存在无穷值的列：", quality_report.filter(pl.col("train_infinite_count") > 0).height)
print("完整报告已保存：", REPORT_ROOT / "feature_quality_report.csv")

display(
    quality_report
    .filter(pl.col("column") != "target")
    .sort("train_missing_rate", descending=True)
    .head(60)
)

正在扫描最终宽表并统计数据质量，Train 表较大，可能需要几分钟……
Train 全空列： 28
Train 缺失率 ≥ 90% 的列： 553
存在无穷值的列： 0
完整报告已保存： /kaggle/working/reports/feature_quality_report.csv


column,train_dtype,train_missing_count,train_missing_rate,train_infinite_count,test_dtype,test_missing_count,test_missing_rate,test_infinite_count,missing_rate_abs_gap,train_all_missing,test_all_missing,train_missing_ge_90pct
str,str,i64,f64,i64,str,i64,f64,i64,f64,bool,bool,bool
"""person_main_childnum_185L""","""Float64""",1526659,1.0,0,"""Float64""",10,1.0,0,0.0,true,true,true
"""person_main_housingtype_772L""","""Null""",1526659,1.0,0,"""Null""",10,1.0,0,0.0,true,true,true
"""person_main_relationshiptoclie…","""Null""",1526659,1.0,0,"""Null""",10,1.0,0,0.0,true,true,true
"""person_main_relationshiptoclie…","""Null""",1526659,1.0,0,"""Null""",10,1.0,0,0.0,true,true,true
"""person_main_remitter_829L""","""Null""",1526659,1.0,0,"""Null""",10,1.0,0,0.0,true,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…
"""person2_person4_addres_role_de…","""Float64""",1526658,0.999999,0,"""Float64""",10,1.0,0,6.5503e-7,false,true,true
"""person2_person4_addres_role_ot…","""Float64""",1526658,0.999999,0,"""Float64""",10,1.0,0,6.5503e-7,false,true,true
"""person2_person4_conts_role_p38…","""Float64""",1526658,0.999999,0,"""Float64""",10,1.0,0,6.5503e-7,false,true,true


## 下游输入

- `home_credit_features_train.parquet`
- `home_credit_features_test.parquet`

以上两份文件可供下游完成特征工程与模型训练。